# Project 3 - NCSU ST 554
#### Author: Trevor Lillywhite
#### Due Date: April 30, 2026

## Introduction

**Purpose:** This project will focus on using `spark` to handle streaming data, fitting a machine learning model, and performing other associated tasks. 

It will involve both this notebook (using a pySpark3 kernel) and producing a `.py` file to read in a stream of data from a static source. After a portion of the dataset is used for model training, the remaining portions will be incrementally streamed in the form of csv files to a folder being monitored by the code. The machine learning model will do prediction on this stream and write the results out to the console.

**Data Source:** The data is adapted from a UCI machine learning dataset on a study predicting power consumption to factors like time of day, temperature, and humidity. More information on the original dataset is available here: 
+ UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city
+ Original Case Study: https://www.semanticscholar.org/paper/Comparison-of-Machine-Learning-Algorithms-for-the-%3A-Salam-Hibaoui/177512e766fe5d8624a1b3e93abd11082ac37b3f


## Fitting the Model

### Reading in Data

The data is saved as a csv in the same directory level as this notebook. However, it can also be accessed at the following URL: 
+ https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv

We will read in the data as a standard `pandas` DataFrame using the `pd.read_csv()` function.

In [2]:
# Import relevant modules
import pandas as pd
import numpy as np
import time
from pyspark.sql.types import StructType
from pyspark.sql import SparkSession

# Create spark session
spark = SparkSession.builder.getOrCreate()

In [3]:
# Read in data to pandas DataFrame
df_pandas = pd.read_csv('power_ml_data.csv', sep=',', header=0)

# Verify data
df_pandas.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [4]:
# Check dataframe info
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47174 entries, 0 to 47173
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Temperature            47174 non-null  float64
 1   Humidity               47174 non-null  float64
 2   Wind_Speed             47174 non-null  float64
 3   General_Diffuse_Flows  47174 non-null  float64
 4   Diffuse_Flows          47174 non-null  float64
 5   Power_Zone_1           47174 non-null  float64
 6   Power_Zone_2           47174 non-null  float64
 7   Power_Zone_3           47174 non-null  float64
 8   Month                  47174 non-null  int64  
 9   Hour                   47174 non-null  int64  
dtypes: float64(8), int64(2)
memory usage: 3.6 MB


It appears that the data was read in properly. All columns are numerical (`float64` or `int64`) with no missing values (47174 non-null values for each column). 

Now we will convert this to a `spark` DataFrame.

In [6]:
# Convert to spark DataFrame
df = spark.createDataFrame(df_pandas)
df.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

The data appears consistent after converting to `spark`. 

Let's check the data types of each column. 

In [7]:
# Check spark DataFrame data types (displayed cleanly in Pandas)
pd.DataFrame(df.dtypes)

,0,1
0,Temperature,double
1,Humidity,double
2,Wind_Speed,double
3,General_Diffuse_Flows,double
4,Diffuse_Flows,double
5,Power_Zone_1,double
6,Power_Zone_2,double
7,Power_Zone_3,double
8,Month,bigint
9,Hour,bigint


Each `float64` value converted to `double` and each `int64` value converted to `bigint`.

### Data Preparation

Our analysis will treat `Power_Zone_3` as the response variable and all other variables as predictors. 

We want to fit an Elastic Net model using cross validation. Because the actual predictions will be performed on a separate streaming dataset, we don't need to do a test/train split on the data we just read in. Data preparation steps will be executed using `MLlib` transformations assembled into a pipeline.

The pipeline will contain following transformations: 
+ Cast `Hour` to `double` type, if not already done. 
+ Binarize the `Hour` column based on the column being less than 6.5 or not (to discriminate nighttime electrical loads from daytime). 
+ One-hot encode the `Month` column.
+ Run a PCA fit on the `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` columns. 
    - This will find a number of aggregated features (using linear combinations of those listed above) that are orthogonal (uncorrelated) and able to be sorted by importance by eigenvalues. 
    - The features will be bundled together using `VectorAssembler()`
    - We will fit the `PCA()` estimator using the feature vector. 
    - Only the two most important Principal Components (PCs) will be used in the pipeline. This will reduce the dimensionality of the data while accounting for most of the variance.
+ The response variable (`Power_Zone_3`) will be renamed to `label`.
+ The two PCs and remaining features will be bundled together using `VectorAssembler()`

In [8]:
# Import relevant modules
from pyspark.sql.functions import col
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, StandardScaler, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

In [9]:
# Create SQL Transformer to cast Hour to double type
sqlCast = SQLTransformer(
    statement = """
                SELECT * EXCEPT (Hour), CAST(Hour AS DOUBLE) 
                FROM __THIS__
                """
)

In [10]:
# Binarize Hour based on threshold of 6.5
binarizer = Binarizer(threshold = 6.5, inputCol = "Hour", outputCol = "Hour_binarized")

In [11]:
# One-hot encode Month column
encoder = OneHotEncoder(inputCols=["Month"], 
                        outputCols=["Month_encoded"])

In [12]:
# Run PCA fit select PCs

# Define features of interest
colsPCA = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"]

# Assemble features into vector
assemblerPCA = VectorAssembler(inputCols = colsPCA, outputCol = "featuresPCA")

# Standardize features: mean=0, std=1. 
scalerPCA = StandardScaler(inputCol = "featuresPCA", 
                           outputCol = "scaledFeaturesPCA", 
                           withStd = True, withMean = True)

# Create PCA transformer
pca = PCA(k=2, inputCol = "scaledFeaturesPCA", outputCol = "pcaFeatures")

Note that after fitting PCA, we can check model.stages[pcaIndex].explainedVariance to see how much information each PC retained

Let's test this and the general PCA fitting/transformation process

In [14]:
# Test PCA fitting and PC importance
pipeTest = Pipeline(stages=[assemblerPCA, scalerPCA, pca])
modelPCA = pipeTest.fit(df)
modelPCA.stages[2].explainedVariance

DenseVector([0.4658, 0.2387])

The PC1 explains nearly half of the variance, and PC2 explains nearly a quarter. 

In the `pca` transformer definition, I also experimented by changing k to equal 5. The remaining components had importances of 0.1438, 0.0837, and 0.0681. It makes sense to only use the first two PCs since they capture about 70% of the total variance and there are diminishing returns after that.

In [15]:
# Complete the PCA fitting test
resultsTest = modelPCA.transform(df)
resultsTestPandas = resultsTest.toPandas()
resultsTestPandas.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,featuresPCA,scaledFeaturesPCA,pcaFeatures
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,"[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.0690743213463727, 0.8078678829472259]"
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,"[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.102921063880655, 0.8152476222806389]"
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,"[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.1120662633791025, 0.8227151924829956]"
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,"[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.143503184742221, 0.8329135817940962]"
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,"[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.1824440036627015, 0.8444681624812327]"


The new columns were produced as directed. The values look credible. 

Let's get back on track with the transformation definitions. 

In [16]:
# Rename response variable as 'label'
sqlRename = SQLTransformer(
    statement = """ 
                SELECT * EXCEPT (Power_Zone_3), (Power_Zone_3) AS label
                FROM __THIS__
                """
)

In [17]:
# Assemble all predictors into vector
assembler = VectorAssembler(inputCols = ["pcaFeatures", "Hour_binarized", "Power_Zone_1", "Power_Zone_2", "Month_encoded"], 
                            outputCol = "features")

In [18]:
# Now assemble into pipeline and check the outputs after fitting
pipelineTransformers = Pipeline(stages = [sqlCast, binarizer, encoder, assemblerPCA, scalerPCA, pca, sqlRename, assembler])

test = pipelineTransformers.fit(df)
test.transform(df).show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+--------------+--------------+--------------------+--------------------+--------------------+-----------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|Hour_binarized| Month_encoded|         featuresPCA|   scaledFeaturesPCA|         pcaFeatures|      label|            features|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+--------------+--------------+--------------------+--------------------+--------------------+-----------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|    1| 0.0|           0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[-2.1079477438809...|[2.06907432134637...|20240.96386|(17,[0,1,3,4,6],[...|
|      6.414|    74.5|     0.083|                 0.07|        0.085

The pipelines appears to have executed correctly. 

### Model Fitting

The pipeline defined above will be used in a `CrossValidator()` function to fit an elastic net estimator using `LinearRegression()` function. We will explore a grid of the following parameters (with all combinations): 
+ `regParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
+ `elasticNetParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

The `CrossValidator()` will use `rmse` as the model selection metric. 

In [19]:
# Instantiate model
en = LinearRegression()

# Add estimator to pipeline
pipeline = Pipeline(stages = [sqlCast, binarizer, encoder, assemblerPCA, scalerPCA, pca, sqlRename, assembler, en])

# Set up parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(en.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(en.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Set up Cross Validator
cv = CrossValidator(estimator = pipeline,
                    estimatorParamMaps = paramGrid,
                    evaluator = RegressionEvaluator(metricName = "rmse"),
                    numFolds = 5,
                    seed = 42,
                    parallelism = 128)

In [20]:
# Fit Cross Validator
enModel = cv.fit(df)

26/04/26 17:42:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/26 17:42:09 WARN Instrumentation: [387c997e] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 17:42:09 WARN Instrumentation: [dafe17a6] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 17:42:10 WARN Instrumentation: [adc82120] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 17:42:10 WARN Instrumentation: [2cbfcfce] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 17:42:10 WARN Instrumentation: [6625b5e8] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 17:42:11 WARN Instrumentation: [d99659e5] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 17:42:12 WARN Instrumentation: [7e5436e9] regP

Note that, without setting the `parallelism` argument, cross-validation calculations are performed fully in series. Here, it took about 12 minutes to explore the hyperparameter space using a single logical processor. After setting `parallelism` to 128, fitting took only 5 minutes. 

Let's check out the optimal values chosen for the tuning parameters and CV error.

In [21]:
# Isolate best estimator and its parameters
bestPipelineModel = enModel.bestModel
bestENModel = bestPipelineModel.stages[-1]
print("Best regParam:", bestENModel.getRegParam())
print("Best elasticNetParam:", bestENModel.getElasticNetParam())

Best regParam: 0.05
Best elasticNetParam: 0.05


The best model had a relatively small amount of regularization overall and also a small amount of L1 regularization (mostly L2 regularization).

Next, we will report the CV error to see how well the fitting process went. These will be averaged across folds.

In [22]:
# Report CV error
print(f"Best CV RMSE: {min(enModel.avgMetrics):.4f} \n")
print(f"RMSE statistics for all parameter combinations (for comparison):")
pd.Series(enModel.avgMetrics).describe()

Best CV RMSE: 2174.9962 

RMSE statistics for all parameter combinations (for comparison):


count     121.000000
mean     2175.008106
std         0.012939
min      2174.996249
25%      2174.997948
50%      2175.000512
75%      2175.016981
max      2175.034889
dtype: float64

There wasn't much variation at all! Based on the min and max, all of the models performed almost identically.

The next step is to report the training set RMSE using the fitted model as a transformer and evaluating on the entire training set. 

In [23]:
# Determine RMSE using full training set
predictions = bestPipelineModel.transform(df)
trainRMSE = RegressionEvaluator().evaluate(predictions)
print(f"Model RMSE on Training Set: {trainRMSE:.4f}")

Model RMSE on Training Set: 2174.4490


This is slightly better than the previous value because the best model was retrained on the entire training set without a hold-out fold.

Lastly, before handling streaming data, we will create a `residual` column (`label` - `prediction`) using the `.withColumn()` method. 

In [24]:
# Create residual column
results = predictions.withColumn("residual", (col("label") - col("prediction")))\
                     .select("residual", "label", "prediction")
results.show()

+------------------+-----------+------------------+
|          residual|      label|        prediction|
+------------------+-----------+------------------+
|-695.2629810062135|20240.96386|20936.226841006213|
| 1429.497871904543|20131.08434| 18701.58646809546|
|1431.2442602262745|19668.43373|18237.189469773726|
|1283.3848937491057|18899.27711|17615.892216250893|
|1425.0277973157936|18442.40964|17017.381842684208|
|1589.5061777940791|18130.12048|16540.614302205922|
| 1830.376006105278|17945.06024| 16114.68423389472|
|1717.2387757975375|17459.27711|15742.038334202462|
|1742.9841969905574|17025.54217|15282.557973009443|
| 1859.317548577077|16794.21687|14934.899321422923|
|1992.8155358608274|16638.07229|14645.256754139173|
|  1999.50190805905|16395.18072| 14395.67881194095|
|2043.8391010806245|16117.59036|14073.751258919376|
| 2213.408500106041| 15822.6506| 13609.24209989396|
|2240.1145750457326|15672.28916|13432.174584954268|
| 2314.823027723127|15597.10843|13282.285402276873|
|2371.431778

## Streaming

In this part, we will use a separate dataset (same rows and initial structure, different observations) and create a `.py` file to simulated a data stream for analysis. The stream will include randomly sampled data from this new dataset. 

### Reading a Stream

The "streamed" data will be sent to a folder called "stream_catcher". The schema for this data will be pulled from the csv header, similar to our method in Homework 10. We will also set up a `readStream`, specifying that the `header` is included. 

In [25]:
# Define schema based on training data header
myschema = df.schema
myschema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

In [26]:
# Create object to store streamed data from folder: 'stream_catcher'
streamed_data = spark.readStream\
                     .schema(myschema)\
                     .option("header", "true")\
                     .csv("stream_catcher")

### Transform/Aggregation Step
With the stream, we will use the model transformer to obtain predictions from the incoming data. We will also create a `residual` column and return only the `label`, `prediction`, and `residual` columns.

Afterwards, we will separately change the raw data stream's response variable to be called `label` so we can `.join()` the results table with the original data! We will do an `inner` join.

In [27]:
# Obtain predictions on incoming data
predictions_stream = bestPipelineModel.transform(streamed_data)

# Create results table with label, prediction, residual
results_stream = predictions_stream.withColumn("residual", (col("label") - col("prediction")))\
                                   .select("residual", "label", "prediction")

In [28]:
# Change response variable name in original stream to 'label'
# Reuse sqlRename transformation (previously defined)
streamed_data_with_label = sqlRename.transform(streamed_data)

### Join and Writiting Step
Per the lecture, the code should perform better when `.start()` is called at the time the `writeStream` method is implemented. Therefore, we will join the streams, write the output to the conolse using the `append` mode, and start the query all in one step. 

In [29]:
# Join the two transformed streams together and start the query
joined_data = results_stream \
                .join(streamed_data_with_label, "label", "inner") \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

26/04/26 17:48:48 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3b055924-4a5b-49b3-836b-cdc9c019de5e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/26 17:48:48 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13492.04819| 3674.959267404507| 9817.088922595492|      4.304|    76.0|     0.082|                0.048|        0.152| 19983.79747| 12342.85714|    1|   7|
|14590.84337| 2383.429768744043|12207.413601255957|      4.805|    76.2|     0.081|                0.059|        0.134| 20421.26582| 12908.20669|    1|   3|
| 14862.6506|1967.0920596337564|12895.558540366244|      4.212|    78.3|     0.081|                0.117|        0.082

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 16106.0241|1319.7781801712063|14786.245919828794|      10.84|    78.4|     0.073|                0.062|        0.156| 23914.93671|  16132.5228|    1|   0|
|14128.19277|1595.6532469750036|12532.539523024996|      11.14|    75.0|     0.081|                 0.07|        0.156| 20725.06329| 13783.58663|    1|   2|
|15307.95181| 1106.313946850887|14201.637863149113|      11.12|    77.9|     0.073|                0.055|        0.145

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|18621.68675|3145.9523822349474|15475.734367765053|      15.41|   58.13|     0.076|                481.1|        43.82| 29140.25316| 19462.61398|    1|  13|
|18373.01205| 2543.851532089815|15829.160517910186|      15.11|   57.86|     0.074|                469.9|        43.43| 29674.93671| 19608.51064|    1|  12|
|16643.85542|  1109.01008786877| 15534.84533213123|      15.59|   55.76|     0.077|                399.4|        40.61

In [31]:
# Stop query when ready
joined_data.stop()

Using a couple test "chunks" of data (placed manually into the monitored folder), we can see that the code works. It is slower than expected for transforming such small packets of data, though. 

### Final Test with Streamed Data

This next section will run the code again, but with a `.py` file writing small csv files (5 observations) to the monitored folder every 10 seconds for 20 iterations.

In [32]:
# Import .py file
import Project3_Lillywhite_DataProducer as DataProducer
import importlib

In [33]:
# Run this to reload the import (if changes were made to .py file)
importlib.reload(DataProducer)

<module 'Project3_Lillywhite_DataProducer' from '/home/jupyter-tflillyw@ncsu.edu/Project3_Lillywhite_DataProducer.py'>

In [34]:
# Join the two transformed streams together and start the query
joined_data = results_stream \
                .join(streamed_data_with_label, "label", "inner") \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

26/04/26 17:52:54 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b6ad3c09-febd-4d7a-b0a1-4a570ac9fb91. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/26 17:52:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [35]:
# Run function in .py file
DataProducer.data_source('power_streaming_data.csv')

iteration 0 complete


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25529.39271|-1649.2364278464556|27178.629137846456|       19.5|    82.5|     4.927|                1.024|         0.89| 45740.06557| 27518.26625|    5|  20|
|16606.45161| -928.8287793373056|17535.280389337306|      11.79|    88.7|      0.08|                129.2|        118.6| 32905.53191| 19328.04878|    3|  15|
|13755.75077|  5705.833028851424| 8049.917741148576|      21.54|   50.25|     4.924|                269.3|       

iteration 2 complete


-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25596.14458| -496.4995408558643|26092.644120855864|      16.31|    77.2|     4.913|                0.048|        0.152| 42938.73418| 25696.04863|    1|  19|
|40548.71473|  6108.842160243752| 34439.87256975625|      25.69|   56.68|     0.072|                13.24|        11.66| 51737.71365| 34958.39493|    8|  19|
|25408.98462|   -1380.6314502358|  26789.6160702358|      20.88|    85.4|     0.067|                0.059|       

iteration 3 complete


...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11618.70968|-1613.2451428567547|13231.954822856755|       7.12|    88.2|     0.092|                136.0|        130.5|     26496.0|  16840.2439|    3|   8|
| 12453.7386|-125.61819538236159|12579.356795382362|      22.15|    65.2|     4.922|                462.2|        199.4| 34156.67396| 19228.63071|   10|  12|
|10971.42857|-1600.3516027240003|   12571.780172724|      16.56|    53.5|     0.082|                534.1|      

iteration 19 complete


-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|22745.07837|-1373.3540002667032|24118.432370266702|      26.44|    71.4|     4.921|                388.8|         63.9| 36855.04994| 23325.87117|    8|  17|
|16506.09231| 1459.2814996072339|15046.810810392766|      22.16|   50.58|      0.08|                873.0|        361.7| 29842.64901| 20353.84615|    6|  10|
|13953.96923| -61.85230767860958| 14015.82153767861|      20.76|    82.6|     4.919|                247.5|      

all iterations complete


-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 23928.3871| 1944.715138334941| 21983.67196166506|      12.17|    81.6|     0.074|                0.051|          0.1| 38052.76596| 26820.73171|    3|  21|
|27805.09091|2008.1008544299366|25796.990055570062|      15.19|    86.6|     0.075|                0.048|        0.115| 42310.26911| 21548.67617|    4|  20|
|27805.09091|2008.1008544299366|25796.990055570062|      15.19|    86.6|     0.075|                0.048|        0.11

In [37]:
# Stop query when ready
joined_data.stop()

Success! The code performed as expected. 

In this project, we created a pipeline with data preparation and model fitting steps. We then created a .py utility to randomly sample a dataset and "stream" small batches of that data to a folder being monitored. We demonstrated the ability to run predictions on that streamed data, separately manipulate the data stream by renaming a variable, and join datastreams that were written out to the console as live output. Overall, we demonstrated the ability to handle streaming data at the basic level using Spark Structured Streaming.  